In [ ]:
from scipy import stats
from scipy.stats import t
import numpy as np
import matplotlib as mpl
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeat
import matplotlib.pyplot as plt
from cartopy.io.shapereader import Reader
import matplotlib.ticker as mticker
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.feature import ShapelyFeature
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import BoundaryNorm
from tqdm import tqdm

In [ ]:

def get_ab_values(rain):
    if  0.<= rain < 2:
        return 0.45131016517318423, 0.16169399717438698
    elif 2 <= rain < 4:
        return  1.6794847980908945, 0.16198899816352366
    elif 4 <= rain < 8:
        return 3.437138732494024, 0.16260406256981108
    elif rain >= 8:
        return 7.550905959233764, 0.16581205841013613
    else:
        return -9999,-9999
    
def func(x, a, b):
    return a * np.exp(-b * x)

In [ ]:
def calc_neff(gauge_vals):
    vals = np.asarray(gauge_vals).flatten()
    n = len(vals)
    # 如果有效值小于2，直接返回
    if n <= 1:
        return n
    # 如果所有值都相同，相关性无意义，直接r=0
    if np.all(vals == vals[0]):
        r = 0
    # 如果只有一个非零，其余全为0，也直接r=0
    elif np.count_nonzero(vals) <= 1:
        r = 0
    else:
        try:
            corr_matrix = np.corrcoef(vals)
            # 如果相关矩阵不是二维，说明输入有问题，直接r=0
            if corr_matrix.ndim < 2:
                r = 0
            else:
                r = np.nanmean(corr_matrix[np.triu_indices(n, k=1)])
                r = 0 if np.isnan(r) else r
        except Exception as e:
            print('corrcoef error:', e, 'vals:', vals)
            r = 0
    neff = n / (1 + (n-1)*r)
    return max(2, neff)

In [ ]:
gauge_pre_num=np.load("../gauge_pre_num.npy")#站点数量,(43848, 35, 55)
df_posi=pd.read_csv('../df_posi_02—10.csv')#站点数
gauge_pre=np.load("../guage_pre_all.npy")#【小时数，站点数】

gauge_pre_grided=np.load('../gauge_pre_grided.npy')#地基格点化降水,(43848, 35, 55)
sate_pre=np.load("../precip_all_hour_satellite.npy")#卫星格点化降水,(43848, 35, 55)

# sate_pre=satellite_pre.transpose(0, 2, 1)
#两个数据同时小于0.1时表示没有误差
sate_pre[(sate_pre<0.1) & (gauge_pre_grided<0.1)]=0#np.nan
gauge_pre_grided[(sate_pre<0.1) & (gauge_pre_grided<0.1)]=0#np.nan
pre_minus = sate_pre - gauge_pre_grided

In [ ]:
# 1. 预处理格点-站点索引
n_rows, n_cols = sate_pre.shape[1:3]
station_map = [[[] for _ in range(n_cols)] for _ in range(n_rows)]
for idx, row in df_posi.iterrows():
    i, j = int(row['indexi']), int(row['indexj'])
    station_map[i][j].append(idx)
    
significance_level=0.05
"""
sate_pre: 卫星格点化降水 (时间, 行, 列)
gauge_pre: 地基原始降水 (时间, 站点数)
df_posi: 站点位置信息，需包含每个站点对应的格点(i,j)
"""
n_time, n_rows, n_cols = sate_pre.shape
t_values = np.full((n_time, n_rows, n_cols), np.nan)
t2_values = np.full((n_time, n_rows, n_cols), np.nan)
p_values = np.full((n_time, n_rows, n_cols), np.nan)
significant_mask = np.full((n_time, n_rows, n_cols), False)
delta_values = np.full((n_time, n_rows, n_cols), np.nan)

In [ ]:
# # delta取0的t检验
# t_values_delta0 = np.full((n_time, n_rows, n_cols), np.nan)
# t2_values_delta0 = np.full((n_time, n_rows, n_cols), np.nan)
# p_values_delta0 = np.full((n_time, n_rows, n_cols), np.nan)
# significant_mask_delta0 = np.full((n_time, n_rows, n_cols), False)
# delta_values_delta0 = np.full((n_time, n_rows, n_cols), np.nan)

# for time in tqdm(range(n_time), desc="时间进度(delta=0)"):
#     for i in range(n_rows):
#         for j in range(n_cols):
#             station_indices = station_map[i][j]
#             if len(station_indices) < 2:
#                 continue
#             gauge_vals = gauge_pre[time, station_indices]
#             gauge_vals = gauge_vals[~np.isnan(gauge_vals)]
#             if len(gauge_vals) < 2:
#                 continue
#             sat_val = sate_pre[time, i, j]
#             if np.isnan(sat_val):
#                 continue
#             gauge_mean = np.mean(gauge_vals)
#             gauge_std = np.std(gauge_vals, ddof=1)
#             n_gauge = len(gauge_vals)
#             delta = 0
#             diff_mean = sat_val - gauge_mean
#             if gauge_std == 0:
#                 if diff_mean != 0:
#                     t_values_delta0[time, i, j] = np.inf
#                     p_values_delta0[time, i, j] = 0.0
#                     significant_mask_delta0[time, i, j] = True
#                     delta_values_delta0[time, i, j] = delta
#                 else:
#                     t_values_delta0[time, i, j] = 0.0
#                     p_values_delta0[time, i, j] = 1.0
#                     significant_mask_delta0[time, i, j] = False
#                     delta_values_delta0[time, i, j] = delta
#                 continue
#             t1 = (diff_mean - delta) / (gauge_std / np.sqrt(n_gauge))
#             t2 = (diff_mean + delta) / (gauge_std / np.sqrt(n_gauge))
#             if n_gauge > 1:
#                 # print(gauge_vals)
#                 neff = calc_neff(gauge_vals)
#                 dfree = max(1, neff - 1)
#             else:
#                 dfree = 0  # 只有一个站点，自由度为0
#             print(dfree)
#             p1 = 1 - t.cdf(t1, dfree)
#             p2 = t.cdf(t2, dfree)
#             p_val = 2 * min(p1, p2)
#             t_values_delta0[time, i, j] = diff_mean
#             p_values_delta0[time, i, j] = p_val
#             significant_mask_delta0[time, i, j] = p_val < significance_level
#             delta_values_delta0[time, i, j] = delta

In [ ]:
# np.save('t_values_delta0.npy', t_values_delta0)
# np.save('t2_values_delta0.npy', t2_values_delta0)
# np.save('p_values_delta0.npy', p_values_delta0)
# np.save('significant_mask_delta0.npy', significant_mask_delta0)
# np.save('delta_values_delta0.npy', delta_values_delta0)

In [ ]:
# # delta放分母的计算
# t_values_delta = np.full((n_time, n_rows, n_cols), np.nan)
# t2_values_delta = np.full((n_time, n_rows, n_cols), np.nan)
# p_values_delta = np.full((n_time, n_rows, n_cols), np.nan)
# significant_mask_delta = np.full((n_time, n_rows, n_cols), False)
# delta_values_delta = np.full((n_time, n_rows, n_cols), np.nan)

# for time in tqdm(range(n_time), desc="时间进度(delta=0)"):
#     for i in range(n_rows):
#         for j in range(n_cols):
#             station_indices = station_map[i][j]
#             if len(station_indices) < 2:
#                 continue
#             gauge_vals = gauge_pre[time, station_indices]
#             gauge_vals = gauge_vals[~np.isnan(gauge_vals)]
#             if len(gauge_vals) < 2:
#                 continue
#             sat_val = sate_pre[time, i, j]
#             if np.isnan(sat_val):
#                 continue
#             gauge_mean = np.mean(gauge_vals)
#             gauge_std = np.std(gauge_vals, ddof=1)
#             n_gauge = len(gauge_vals)
#             delta = func(n_gauge, get_ab_values(gauge_mean)[0], get_ab_values(gauge_mean)[1])
#             diff_mean = sat_val - gauge_mean
#             if gauge_std == 0:
#                 if diff_mean != 0:
#                     t_values_delta[time, i, j] = np.inf
#                     p_values_delta[time, i, j] = 0.0
#                     significant_mask_delta[time, i, j] = True
#                     delta_values_delta[time, i, j] = delta
#                 else:
#                     t_values_delta[time, i, j] = 0.0
#                     p_values_delta[time, i, j] = 1.0
#                     significant_mask_delta[time, i, j] = False
#                     delta_values_delta[time, i, j] = delta
#                 continue
#             t1 = (diff_mean - 0) / (delta )
#             t2 = (diff_mean + 0) / (delta )
#             if n_gauge > 1:
#                 # print(gauge_vals)
#                 neff = calc_neff(gauge_vals)
#                 dfree = max(1, neff - 1)
#             else:
#                 dfree = 0  # 只有一个站点，自由度为0
#             # print(dfree)
#             p1 = 1 - t.cdf(t1, dfree)
#             p2 = t.cdf(t2, dfree)
#             p_val = 2 * min(p1, p2)
#             t_values_delta[time, i, j] = diff_mean
#             p_values_delta[time, i, j] = p_val
#             significant_mask_delta[time, i, j] = p_val < significance_level
#             delta_values_delta[time, i, j] = delta

In [ ]:
# np.save('t_values_delta.npy', t_values_delta)
# np.save('t2_values_delta.npy', t2_values_delta)
# np.save('p_values_delta.npy', p_values_delta)
# np.save('significant_mask_delta.npy', significant_mask_delta)
# np.save('delta_values_delta.npy', delta_values_delta)

In [ ]:
# ##容错

# # 2. 主循环
# for time in tqdm(range(n_time), desc="时间进度"):
#     for i in range(n_rows):
#         for j in range(n_cols):
#             station_indices = station_map[i][j]
#             if len(station_indices) < 2:
#                 continue
#             gauge_vals = gauge_pre[time, station_indices]
#             gauge_vals = gauge_vals[~np.isnan(gauge_vals)]
#             if len(gauge_vals) < 2:
#                 continue
#             sat_val = sate_pre[time, i, j]
#             if np.isnan(sat_val):
#                 continue
#             gauge_mean = np.mean(gauge_vals)
#             gauge_std = np.std(gauge_vals, ddof=1)
#             n_gauge = len(gauge_vals)
#             delta = func(n_gauge, get_ab_values(gauge_mean)[0], get_ab_values(gauge_mean)[1])
#             diff_mean = sat_val - gauge_mean
#             if gauge_std == 0:
#                 if diff_mean != 0:
#                     t_values[time, i, j] = np.inf
#                     p_values[time, i, j] = 0.0
#                     significant_mask[time, i, j] = True
#                     delta_values[time, i, j] = delta
#                 else:
#                     t_values[time, i, j] = 0.0
#                     p_values[time, i, j] = 1.0
#                     significant_mask[time, i, j] = False
#                     delta_values[time, i, j] = delta
#                 continue
#             t1 = (diff_mean - delta) / (gauge_std / np.sqrt(n_gauge))
#             t2 = (diff_mean + delta) / (gauge_std / np.sqrt(n_gauge))
#             if n_gauge > 2:
#                 neff = calc_neff(gauge_vals)
#                 dfree = max(1, neff - 1)
#             else:
#                 dfree = n_gauge - 1
#             p1 = 1 - t.cdf(t1, dfree)
#             p2 = t.cdf(t2, dfree)
#             p_val = 2 * min(p1, p2)
#             t_values[time, i, j] = diff_mean
#             p_values[time, i, j] = p_val
#             significant_mask[time, i, j] = p_val < significance_level
#             delta_values[time, i, j] = delta

In [ ]:
# np.save('t_values.npy', t_values)
# np.save('t2_values.npy', t2_values)
# np.save('p_values.npy', p_values)
# np.save('significant_mask.npy', significant_mask)
# np.save('delta_values.npy', delta_values)

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

# 1. 拉平
p_flat = p_values.reshape(-1)
valid_mask = ~np.isnan(p_flat)
p_valid = p_flat[valid_mask]

# 2. FDR校正
rej, pval_corr, _, _ = multipletests(p_valid, alpha=0.05, method='fdr_bh')

# 3. 还原
significant_mask_fdr = np.full_like(p_flat, False, dtype=bool)
significant_mask_fdr[valid_mask] = rej
significant_mask_fdr = significant_mask_fdr.reshape(p_values.shape)

In [ ]:

# 1. 拉平
p_flat = p_values_delta0.reshape(-1)
valid_mask = ~np.isnan(p_flat)
p_valid = p_flat[valid_mask]

# 2. FDR校正
rej, pval_corr, _, _ = multipletests(p_valid, alpha=0.05, method='fdr_bh')

# 3. 还原
significant_mask_fdr_delta0 = np.full_like(p_flat, False, dtype=bool)
significant_mask_fdr_delta0[valid_mask] = rej
significant_mask_fdr_delta0 = significant_mask_fdr_delta0.reshape(p_values_delta0.shape)

In [ ]:
significant_mask.shape

In [ ]:
hours_per_year = [8784, 8760, 8760, 8760, 8784]  # 5年
def get_season_indices(year_idx, is_leap):
    base = sum(hours_per_year[:year_idx])
    leap_offset = 24 if is_leap else 0
    return {
        "spring": (
            base + (31*24 + 28*24 + leap_offset),
            base + (31*24 + 28*24 + 92*24 + leap_offset)
        ),
        "summer": (
            base + (31*24 + 28*24 + 92*24 + leap_offset),
            base + (31*24 + 28*24 + 184*24 + leap_offset)
        ),
        "autumn": (
            base + (31*24 + 28*24 + 184*24 + leap_offset),
            base + (31*24 + 28*24 + 275*24 + leap_offset)
        ),
        "winter": [
            (base + 334*24, base + (366 if is_leap else 365)*24),
            (base, base + 59*24 + (24 if is_leap else 0))
        ]
    }

In [ ]:
significant_mask_delta0=np.load(r'.\t_test_without_dfree_change\significant_mask_delta0.npy')
significant_mask=np.load(r'.\t_test_without_dfree_change\significant_mask.npy')
significant_mask_delta=np.load(r'significant_mask_delta.npy')

In [ ]:
#计算四季地基降水
season_gauge_4d = []
for i in range(4):
    season_gauge_4d.append([])

for year in range(5):
    is_leap = (hours_per_year[year] == 8784)
    indices = get_season_indices(year, is_leap)
    for i, season in enumerate(["spring", "summer", "autumn"]):
        start, end = indices[season]
        season_gauge_4d[i].append(gauge_pre_grided[start:end])
    # 冬季特殊处理
    winter12 = gauge_pre_grided[indices["winter"][0][0]:indices["winter"][0][1]]
    winter1_2 = gauge_pre_grided[indices["winter"][1][0]:indices["winter"][1][1]]
    season_gauge_4d[3].append(np.concatenate([winter1_2, winter12]))

# 合并年份
for i in range(4):
    season_gauge_4d[i] = np.concatenate(season_gauge_4d[i], axis=0)  # shape: (总小时数, 35, 55)

In [ ]:
season_mask = [[] for _ in range(4)]
for year in range(5):
    is_leap = (hours_per_year[year] == 8784)
    indices = get_season_indices(year, is_leap)
    for i, season in enumerate(["spring", "summer", "autumn"]):
        start, end = indices[season]
        season_mask[i].append(significant_mask[start:end])#_fdr
    # 冬季特殊处理
    winter12 = significant_mask[indices["winter"][0][0]:indices["winter"][0][1]]
    winter1_2 = significant_mask[indices["winter"][1][0]:indices["winter"][1][1]]
    season_mask[3].append(np.concatenate([winter1_2, winter12]))
# 合并年份
season_mask_4d = []
for i in range(4):
    season_mask_4d.append(np.concatenate(season_mask[i], axis=0))  # shape: (总小时数, 35, 55)

proportion_season = []
for i in range(4):
    mask_rain=season_gauge_4d[i]>0
    mask_sig = season_mask_4d[i]
    # 统计有降水的样本数
    count_rain = np.sum(mask_rain, axis=0)  # (行, 列)
    # 统计有降水且显著的样本数
    count_sig = np.sum(mask_rain & mask_sig, axis=0)  # (行, 列)
    proportion_sig_rain = count_sig / count_rain
    proportion_sig_rain[count_rain == 0] = np.nan
    proportion_sig_rain[proportion_sig_rain==0]=np.nan
    proportion_season.append(proportion_sig_rain)  # (35, 55)

# proportion_season = []
# for i in range(4):
#     temp=np.mean(season_mask_4d[i], axis=0)
#     temp[temp==0]=np.nan
#     proportion_season.append(temp) 

In [ ]:
season_mask_delta = [[] for _ in range(4)]
for year in range(5):
    is_leap = (hours_per_year[year] == 8784)
    indices = get_season_indices(year, is_leap)
    for i, season in enumerate(["spring", "summer", "autumn"]):
        start, end = indices[season]
        season_mask_delta[i].append(significant_mask_delta[start:end])#_fdr
    # 冬季特殊处理
    winter12 = significant_mask_delta[indices["winter"][0][0]:indices["winter"][0][1]]
    winter1_2 = significant_mask_delta[indices["winter"][1][0]:indices["winter"][1][1]]
    season_mask_delta[3].append(np.concatenate([winter1_2, winter12]))
# 合并年份
season_mask_4d_delta = []
for i in range(4):
    season_mask_4d_delta.append(np.concatenate(season_mask_delta[i], axis=0))  # shape: (总小时数, 35, 55)

proportion_season_delta = []
for i in range(4):
    mask_rain=season_gauge_4d[i]>0
    mask_sig = season_mask_4d_delta[i]
    # 统计有降水的样本数
    count_rain = np.sum(mask_rain, axis=0)  # (行, 列)
    # 统计有降水且显著的样本数
    count_sig = np.sum(mask_rain & mask_sig, axis=0)  # (行, 列)
    proportion_sig_rain = count_sig / count_rain
    proportion_sig_rain[count_rain == 0] = np.nan
    proportion_sig_rain[proportion_sig_rain==0]=np.nan
    proportion_season_delta.append(proportion_sig_rain)  # (35, 55)


# proportion_season_delta0 = []
# for i in range(4):
#     temp=np.mean(season_mask_4d_delta0[i], axis=0)
#     temp[temp==0]=np.nan
#     proportion_season_delta0.append(temp) 

In [ ]:
season_mask_delta0 = [[] for _ in range(4)]
for year in range(5):
    is_leap = (hours_per_year[year] == 8784)
    indices = get_season_indices(year, is_leap)
    for i, season in enumerate(["spring", "summer", "autumn"]):
        start, end = indices[season]
        season_mask_delta0[i].append(significant_mask_delta0[start:end])#_fdr
    # 冬季特殊处理
    winter12 = significant_mask_delta0[indices["winter"][0][0]:indices["winter"][0][1]]
    winter1_2 = significant_mask_delta0[indices["winter"][1][0]:indices["winter"][1][1]]
    season_mask_delta0[3].append(np.concatenate([winter1_2, winter12]))
# 合并年份
season_mask_4d_delta0 = []
for i in range(4):
    season_mask_4d_delta0.append(np.concatenate(season_mask_delta0[i], axis=0))  # shape: (总小时数, 35, 55)

proportion_season_delta0 = []
for i in range(4):
    mask_rain=season_gauge_4d[i]>0
    mask_sig = season_mask_4d_delta0[i]
    # 统计有降水的样本数
    count_rain = np.sum(mask_rain, axis=0)  # (行, 列)
    # 统计有降水且显著的样本数
    count_sig = np.sum(mask_rain & mask_sig, axis=0)  # (行, 列)
    proportion_sig_rain = count_sig / count_rain
    proportion_sig_rain[count_rain == 0] = np.nan
    proportion_sig_rain[proportion_sig_rain==0]=np.nan
    proportion_season_delta0.append(proportion_sig_rain)  # (35, 55)


# proportion_season_delta0 = []
# for i in range(4):
#     temp=np.mean(season_mask_4d_delta0[i], axis=0)
#     temp[temp==0]=np.nan
#     proportion_season_delta0.append(temp) 

In [ ]:
for i in range(4):
    print(np.nanmean(proportion_season_delta0[i])-np.nanmean(proportion_season_delta[i]))

In [ ]:
num_fla=gauge_pre_num.mean(axis=0).flatten()
for i in range(4):
    prop_delta0 = proportion_season_delta0[i].flatten()
    prop_season = proportion_season_delta[i].flatten()
    valid_mask = ~(np.isnan(num_fla) | np.isnan(prop_delta0) | np.isnan(prop_season))
    
    # 提取有效数据
    num_fla_valid = num_fla[valid_mask]
    prop_delta0_valid = prop_delta0[valid_mask]
    prop_season_valid = prop_season[valid_mask]
    corr_delta0 = np.corrcoef(num_fla_valid, prop_delta0_valid)[0, 1]
    corr_season = np.corrcoef(num_fla_valid, prop_season_valid)[0, 1]
    print(f"季节 {i}: {corr_delta0},{corr_season},{corr_delta0 - corr_season}")

In [ ]:
#分母为总降水
# season_mask_delta0 = [[] for _ in range(4)]
# for year in range(5):
#     is_leap = (hours_per_year[year] == 8784)
#     indices = get_season_indices(year, is_leap)
#     for i, season in enumerate(["spring", "summer", "autumn"]):
#         start, end = indices[season]
#         season_mask_delta0[i].append(significant_mask_delta0[start:end])
#     # 冬季特殊处理
#     winter12 = significant_mask_delta0[indices["winter"][0][0]:indices["winter"][0][1]]
#     winter1_2 = significant_mask_delta0[indices["winter"][1][0]:indices["winter"][1][1]]
#     season_mask_delta0[3].append(np.concatenate([winter1_2, winter12]))
# # 合并年份
# season_mask_4d_delta0 = []
# for i in range(4):
#     season_mask_4d_delta0.append(np.concatenate(season_mask_delta0[i], axis=0))  # shape: (总小时数, 35, 55)

# proportion_season_delta0 = []
# for i in range(4):
#     temp=np.mean(season_mask_4d_delta0[i], axis=0)
#     temp[temp==0]=np.nan
#     proportion_season_delta0.append(temp) 

In [ ]:
plt.rcParams["font.sans-serif"] = ["Arial"]  # 用于显示中文,Arial
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False  # 用于显示中文

# --设置shp路径，数据集已公开
shp_path = r"E:\0000000000\map_data\bou2_4p.dbf"
# --设置tif路径，数据集已公开
tif_path = r"E:\0000000000\map_data\地形数据\NE1_50M_SR_W.tif"

bins = np.arange(0,0.91,0.01)#,np.arange(2,2.5,0.01))
#bins = np.arange(0,0.2,0.01)
nbin = len(bins) + 1
cmap1 =plt.get_cmap("Spectral_r", nbin)
norm1 = mpl.colors.BoundaryNorm(bins, nbin,extend='both')

# bins = np.logspace(0, np.log10(50), num=100)  # 使用对数刻度范围
# nbin = len(bins)
# cmap1 = plt.get_cmap("rainbow", nbin)

# # 使用 LogNorm 进行颜色归一化
# norm1 = mpl.colors.LogNorm(vmin=1e-1, vmax=50)  # 对数归一化，确保数据范围有效

bins = np.arange(0,0.91,0.01)
#bins = np.arange(0,0.2,0.01)
nbin = len(bins) + 1
cmap2 = plt.get_cmap("Spectral_r", nbin)
norm2 = mpl.colors.BoundaryNorm(bins, nbin,extend='both')

In [ ]:

fig, axs = plt.subplots(2, 4, figsize=(12, 4), dpi=400,subplot_kw={'projection': ccrs.PlateCarree()}#,gridspec_kw={'hspace': 0.05 }#'wspace': 0.0,
                        )
axs=axs.ravel()
for i in range(8):
    ax = axs[i]
    ax.coastlines()
    
    ax.patch.set_facecolor("lightgrey") 
    provinces = ShapelyFeature(Reader(shp_path).geometries(), ccrs.PlateCarree(), edgecolor="k", facecolor="none")
    ax.add_feature(provinces, lw=0.6, zorder=2)
    
    ax.set_extent([111, 122, 30, 37], crs=ccrs.PlateCarree())
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.8, color="gray", linestyle=":")
    gl.top_labels, gl.bottom_labels, gl.right_labels, gl.left_labels = False, False, False, False
    gl.xlocator = mticker.FixedLocator(np.arange(111, 122, 0.5))
    gl.ylocator = mticker.FixedLocator(np.arange(30, 37, 0.5))
    
    if i==0 or i==4  :
        ax.set_yticks(np.arange(30, 38, 2), crs=ccrs.PlateCarree())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(labelcolor="k", length=2, labelsize=10,rotation=0,pad=2)

    if i in [4,5,6,7]  :
    # if i in [0,1,2,3]  :
        ax.set_xticks(np.arange(111, 123, 3), crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.tick_params(labelcolor="k", length=2, labelsize=10,rotation=0,pad=2)

    if i in [0,1,2,3]:
        im = ax.imshow(proportion_season_delta0[i],#alpha=colors[i,j],
            cmap=cmap1,
            norm=norm1,
            origin="lower",
            extent=(111, 122, 30, 37),
        )
    if i in [4,5,6,7]:
        im = ax.imshow(proportion_season_delta[i-4],#alpha=colors[i,j],
            cmap=cmap2,
            norm=norm2,
            origin="lower",
            extent=(111, 122, 30, 37),
        )
    if i==3:
        cbar_ax = fig.add_axes([0.92, 0.2, 0.015, 0.6])
        cbar=plt.colorbar (im,cax=cbar_ax,extend='max',orientation='vertical',ticks=np.arange(0,1.5,0.1))
        font = {#'family' : 'serif',
                        #'color'  : 'darkred',
                        #'weight' : 'normal',
                        'size'   : 10,}
        # cbar.ax.set_title('mm/h',fontdict=font)
        cbar.ax.tick_params(labelsize=13)
        cbar.minorticks_off()

axs[0].set_title('(a) Fraction in Spring',fontsize=10,loc='left')
axs[1].set_title('(b) Fraction in Summer',fontsize=10,loc='left')
axs[2].set_title('(c) Fraction in Fall',fontsize=10,loc='left')
axs[3].set_title('(d) Fraction in Winter',fontsize=10,loc='left')
axs[4].set_title('(e) Corrected Fraction in Spring',fontsize=10,loc='left')
axs[5].set_title('(f) Corrected Fraction in Summer',fontsize=10,loc='left')
axs[6].set_title('(g) Corrected Fraction in Fall',fontsize=10,loc='left')
axs[7].set_title('(h) Corrected Fraction in Winter',fontsize=10,loc='left')
# axs[0].annotate('(a)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[1].annotate('(b)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[2].annotate('(c)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[3].annotate('(d)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[4].annotate('(e)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[5].annotate('(f)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[6].annotate('(g)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[7].annotate('(h)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)

# axs[0].set_ylabel('RMSE$^B$_max',fontsize=12)
# axs[4].set_ylabel('RMSE$^B$_min',fontsize=12)

plt.show()